## 0. Setup (run first in every notebook)
Attach these inputs via **Add Input**: your uploaded `causal-rag-kaggle` dataset (this project) and, depending on the stage,
the Lending Club dataset (`wordsforthewise/lending-club`) and/or the artifacts dataset saved from the previous notebook.
Turn **Internet ON** (policy documents, models, RCT data) and set the accelerator to **GPU T4 x2** for stages 04-08.

In [ ]:
!pip -q install causal-learn econml sentence-transformers bitsandbytes accelerate gradio pypdf beautifulsoup4 scikit-uplift pyarrow 2>&1 | tail -1
import sys, os, glob
hits = glob.glob('/kaggle/input/**/causal_rag/__init__.py', recursive=True)
print('causal_rag copies found:'); [print('  ', h) for h in hits]
need = ('data_prep.py', 'causal_graph.py', 'retrieval.py', 'counterfactual.py', 'audit.py', 'compare.py', 'pipeline.py', 'live_demo.py')
complete = [h for h in hits if all(os.path.exists(os.path.join(os.path.dirname(h), n)) for n in need)]
v11 = [h for h in complete if 'prompt_graded' in open(os.path.join(os.path.dirname(h), 'prompts.py')).read()]
if not complete:
    for h in hits: print('FILES in', os.path.dirname(h), '->', sorted(os.listdir(os.path.dirname(h))))
assert complete, "No COMPLETE causal_rag package found in the attached dataset: re-upload causal_rag_kaggle.zip as a new dataset version."
assert v11, "The mounted package is an OLD version (prompts.py has no prompt_graded; need code 1.3.0). Upload the latest causal_rag_kaggle.zip as a NEW VERSION of the dataset, STOP the session (power icon), re-attach the latest version, and start again."
pkg = v11[0]
for m in [m for m in sys.modules if m.startswith('causal_rag')]: del sys.modules[m]   # drop any stale import
sys.path.insert(0, os.path.dirname(os.path.dirname(pkg)))
print('USING:', pkg)
from causal_rag import paths
paths.restore_artifacts()          # pulls artifacts from any attached output-dataset of an earlier stage
SMOKE   = False                    # True = tiny REAL-data dry run (minutes) to verify the pipeline
N_EVAL  = 12 if SMOKE else 600     # applicants scored by the LLM systems (v1.0/v1.1 used 300; ~5 h at 600, ~6.5 h at 800)
CHUNK   = 6  if SMOKE else 100     # LLM stage saves every CHUNK applicants and resumes after a crash
RUN_NAME = 'v1_2_run'               # name of this run's folder in the exported results
AUDIT_SYSTEM = 'causal_rag_smooth'  # system used by the audit agent: 'causal_rag' (v1.1 prompt) or 'causal_rag_smooth'
EPOCHS  = 3  if SMOKE else 40
N_REPORT= 3  if SMOKE else 10      # applicants that get full DAG + HTML + rationale
WITH_ECONML = False                # EconML causal forest is very slow (hours on CPU); turn on only if you have the time
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'NONE - CPU ONLY')
REQUIRE_GPU = True                 # stages 03-08 need a GPU: on CPU, embeddings alone took ~6 h and CALF ~2.5 h
assert HAS_GPU or not REQUIRE_GPU or SMOKE, ("No GPU in this session. Settings -> Accelerator -> GPU T4 x2 "
                                             "(or your GPU quota is used up). Refusing to start a multi-hour CPU run.")
print('artifacts dir:', paths.ART, '| SMOKE =', SMOKE)

# LIVE DEMO - Causal-RAG audit on real 2015 loans (Gradio)
Starts a public Gradio link from this notebook session. Needs: GPU, Internet ON, the code dataset, and the **artifacts dataset** from a finished run
(`lc_clean.parquet`, `calf_models.pt`, `featurizers.pkl`, `calf_scores.parquet`, `case_emb.npy`, `case_cards.parquet`, `policy_chunks.parquet`,
`policy_emb.npy`, `graph.json`, `adjustment_sets.json`). Nothing is retrained: loading takes a few minutes, each audit about 15-30 s.

In [ ]:
import json, numpy as np, pandas as pd
from causal_rag import retrieval as rt, embedding, causal_graph as cgm, pipeline as pl, live_demo
from causal_rag.llm import LLM
from causal_rag.config import fit_medians, add_features
need = ['lc_clean.parquet','calf_models.pt','featurizers.pkl','calf_scores.parquet','case_emb.npy','case_cards.parquet',
        'policy_chunks.parquet','policy_emb.npy','graph.json','adjustment_sets.json']
missing = [n for n in need if not paths.have(n)]
assert not missing, f"Missing artifacts: {missing}. Attach the dataset made from your run's artifacts/ folder."
df = pd.read_parquet(paths.art('lc_clean.parquet')); tr = (df.role=='train').values
medians = fit_medians(df, tr); d = add_features(df, medians)
idx = d[d.role.isin(['train','val'])].reset_index(drop=True)
assert (idx.case_id.values == pd.read_parquet(paths.art('case_cards.parquet')).case_id.values).all(), 'case index misaligned'
emb = embedding.Embedder()
g, adj = cgm.load_graph()
R = rt.RetrieverAgent(idx, np.load(paths.art('case_emb.npy')).astype('float32'), pd.read_parquet(paths.art('policy_chunks.parquet')),
                      np.load(paths.art('policy_emb.npy')).astype('float32'),
                      pd.read_parquet(paths.art('calf_scores.parquet')).set_index('case_id'), emb, adj)
llm = LLM()
SYSTEM = 'causal_graded'                                  # audit system used by the demo
thr = band = None
if paths.have('llm_scores_v13.parquet'):
    s13 = pd.read_parquet(paths.art('llm_scores_v13.parquet'))
    if SYSTEM in s13.columns:
        m = s13[SYSTEM].values; thr, band = float(np.quantile(m, .8)), float(.15*m.std()); print('operating point from saved scores:', thr, band)
test_df = d[d.role=='test'].reset_index(drop=True)
aud = live_demo.LiveAuditor(llm, R, pl.CalfScorer(), g, test_df, medians, thr, band, system=SYSTEM)
out = aud.analyze(aud.applicant(aud.case_ids()[0]))      # self-test + warm-up
print('self-test OK ->', out['rec']['decision'], '| flags:', out['rec']['flags'], '| operating point:', round(aud.thr,3), round(aud.band,3))

In [ ]:
ui = live_demo.build_ui(aud)
ui.launch(share=True, debug=True)    # prints a public *.gradio.live link; this cell keeps running while the demo is up (stop it to shut down)